# Basis Transforms

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook transforms vector and metric components between spherical and
Cartesian bases, then validates the transforms with symbolic residuals.

Navigation: [Index][index] | Previous: [Reference-Metric Applications][prev]
| Next: [Curvilinear Wave Equation][next]

[index]: ../index.ipynb
[prev]: reference_metric_applications.ipynb
[next]: ../3-wave_equation/wave_equation_curvilinear.ipynb

### Required Reading

- [Reference-Metric Applications](reference_metric_applications.ipynb)
- [Reference Metrics](../1-intro/reference_metric.ipynb)

### NRPy Source Code

- [Reference metrics](https://github.com/nrpy/nrpy/blob/main/nrpy/reference_metric.py): provides the
  spherical reference metric used as the source geometry.
- [Basis-transform Jacobians][basis-source]: builds the Jacobians that convert
  vector and tensor components between reference-metric and Cartesian bases.

[basis-source]: https://github.com/nrpy/nrpy/blob/main/nrpy/equations/basis_transforms/jacobians.py

# Table of Contents

1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Notation and Terms](#Notation-and-Terms)
1. [Step 1](#Step-1:-Import-Tools): Import the symbolic and NRPy tools.
1. [Step 2](#Step-2:-Build-the-Spherical-Transform-Objects): Build the
   transform helper.
1. [Step 3](#Step-3:-Transform-a-Radial-Vector): Transform a vector.
1. [Validation Check](#Validation-Check): Check vector and metric residuals.
1. [Learning Check](#Learning-Check)
1. [Continue](#Continue)

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **Basis:** the local directions used to describe components of a vector or
  tensor.
- **Component:** one number in a vector or tensor description.
- **Reference metric:** the coordinate-system geometry used as the baseline
  for curvilinear calculations.
- **Jacobian:** a table of coordinate derivatives that converts components
  between bases.
- **Round trip:** transforming to another basis and then back again.
- **Residual:** the difference between a computed expression and a trusted
  expression.
- **Contravariant vector:** a vector with upper-index components, written
  here as `V^i`.
- **Covariant rank-2 tensor:** a two-index lower-index object, written here
  as `T_{ij}` or the metric `gamma_{ij}`.
- **Symmetric rank-2 tensor:** a tensor with `T_{ij} = T_{ji}`. In three
  dimensions it has nine stored matrix entries but six independent entries.

# Notation and Terms
### [Back to [top](#Table-of-Contents)]

Spherical coordinates map to Cartesian coordinates as

$$
x = r \sin(\theta)\cos(\phi), \quad
y = r \sin(\theta)\sin(\phi), \quad
z = r \cos(\theta).
$$

Indices `i`, `j`, `a`, and `b` run over `0, 1, 2`. The coordinate basis is
spherical for `q^a = (r, theta, phi)` and Cartesian for `x^i = (x, y, z)`.
When one upper and one lower index are repeated in the equations below, the
repeated index is summed over `0, 1, 2`.

A contravariant vector transforms with the Jacobian:

$$
V^i_{\rm Cart} = \frac{\partial x^i}{\partial q^j} V^j_{\rm rfm}.
$$

A covariant rank-2 tensor such as the reference metric transforms as

$$
\gamma^{\rm Cart}_{ij}
= \frac{\partial q^a}{\partial x^i}
  \frac{\partial q^b}{\partial x^j}
  \gamma^{\rm rfm}_{ab}.
$$

NRPy stores vectors as Python lists, so `V[0]`, `V[1]`, and `V[2]` are the
three components. It stores rank-2 tensors as nested lists, so `T[0][1]` is
the component with first index `0` and second index `1`.

| Mathematical object | Python object | Role | Evidence |
| --- | --- | --- | --- |
| spherical reference metric | `spherical` | source geometry | scale factors |
| transform helper | `transforms` | applies Jacobians | residual checks |
| radial vector | `radial_vectorU` | first example vector | Cartesian output |
| Cartesian vector | `cartesian_vectorU` | transformed vector | components |
| generic vector | `generic_vectorU` | round-trip test input | zero residuals |
| Cartesian metric | `cartesian_metricDD` | transformed metric | identity check |

# Step 1: Import Tools
### [Back to [top](#Table-of-Contents)]

This setup cell imports SymPy and the NRPy modules used below. It has no
output because the first visible evidence appears when we instantiate the
spherical reference metric.

In [1]:
import sympy as sp

import nrpy.indexedexp as ixp
import nrpy.reference_metric as refmetric
from nrpy.equations.basis_transforms.jacobians import BasisTransforms

# Step 2: Build the Spherical Transform Objects
### [Back to [top](#Table-of-Contents)]

The reference metric stores the coordinate geometry. The transform helper
uses that geometry to convert vector and tensor components between the
spherical reference-metric basis and the Cartesian basis.

Inspect the output for:

- the coordinate-system name;
- the three spherical scale factors;
- the `r` and `sin(theta)` factors that later affect vector and tensor
  components.

In [2]:
spherical = refmetric.reference_metric["Spherical"]
transforms = BasisTransforms("Spherical")
print("reference metric:", spherical.CoordSystem)
print("scale factors:", spherical.scalefactor_orthog)

Setting up reference_metric[Spherical]...


reference metric: Spherical
scale factors: [1, xx0, xx0*sin(xx1)]


# Step 3: Transform a Radial Vector
### [Back to [top](#Table-of-Contents)]

The vector below has only a radial spherical-basis component. In Cartesian
components it should become `V_r` times the usual unit-radial direction from
the coordinate map.

Inspect the three printed components as the `x`, `y`, and `z` pieces of that
unit-radial direction.

In [3]:
radial_amplitude = sp.Symbol("V_r", real=True)
radial_vectorU = [radial_amplitude, sp.sympify(0), sp.sympify(0)]
cartesian_vectorU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    radial_vectorU
)
print("Cartesian components of a radial vector:")
for component in cartesian_vectorU:
    print(sp.trigsimp(component))

Cartesian components of a radial vector:
V_r*sin(xx1)*cos(xx2)
V_r*sin(xx1)*sin(xx2)
V_r*cos(xx1)


# Validation Check
### [Back to [top](#Table-of-Contents)]

The first validation uses a generic vector. The trusted result is that a
transform from spherical to Cartesian and back returns the original three
components. The newly computed result is the round-trip vector produced by
`BasisTransforms`.

In [4]:
generic_vectorU = ixp.declarerank1("V")
cartesian_genericU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    generic_vectorU
)
round_trip_vectorU = transforms.basis_transform_vectorU_from_Cartesian_to_rfmbasis(
    cartesian_genericU
)
round_trip_residuals = [
    sp.trigsimp(sp.simplify(round_trip_vectorU[i] - generic_vectorU[i]))
    for i in range(3)
]
print("component | vector round-trip residual")
for i, residual in enumerate(round_trip_residuals):
    print(i, "|", residual)
if any(residual != 0 for residual in round_trip_residuals):
    raise RuntimeError("Expected all vector round-trip residuals to vanish.")
print("validation status: vector round trip passed")

component | vector round-trip residual
0 | 0
1 | 0
2 | 0
validation status: vector round trip passed


The second validation transforms the spherical reference metric to Cartesian
components. The trusted result is the identity matrix for flat Euclidean
space in Cartesian coordinates. The newly computed result is
`cartesian_metricDD`.

This check covers all nine stored matrix entries. For a symmetric metric this
includes all six independent entries and their symmetric partners.

In [5]:
cartesian_metricDD = transforms.basis_transform_tensorDD_from_rfmbasis_to_Cartesian(
    transforms.rfm.ghatDD
)
metric_residuals = []
print("component | Cartesian metric residual")
for i in range(3):
    for j in range(3):
        expected = sp.sympify(1) if i == j else sp.sympify(0)
        residual = sp.trigsimp(sp.simplify(cartesian_metricDD[i][j] - expected))
        metric_residuals.append(residual)
        print(f"({i}, {j}) |", residual)
if any(residual != 0 for residual in metric_residuals):
    raise RuntimeError("Expected all transformed metric residuals to vanish.")
print("validation status: Cartesian metric identity passed")

component | Cartesian metric residual


(0, 0) | 0


(0, 1) | 0
(0, 2) | 0
(1, 0) | 0


(1, 1) | 0
(1, 2) | 0
(2, 0) | 0
(2, 1) | 0
(2, 2) | 0
validation status: Cartesian metric identity passed


The radial-vector components show how the same geometric vector is described
in a different basis. The zero vector round-trip residuals show that the
component transform can be undone. The zero metric residuals show that the
spherical reference metric becomes the Cartesian identity metric after the
tensor transform.

# Learning Check
### [Back to [top](#Table-of-Contents)]

Choose one printed Cartesian component of the radial vector and connect it to
the spherical coordinate map. Then explain why the residual tables validate
the transform more directly than visual similarity between formulas.

# Continue
### [Back to [top](#Table-of-Contents)]

- [Reference-Metric Applications](reference_metric_applications.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)
- [Relativity Variable Conversions](../6-numerical_relativity/adm_bssn_conversions.ipynb)